In [1]:
# =========================================================
# PS S6E4 - exp_20260410_040_safe_blend_compare_core3
# Safe blend comparison for core 3 models:
# 018 / 025 / 026
#
# Strategies:
# - single baselines
# - equal average
# - ridge
# - Nelder-Mead
# - greedy hill climbing
# - LogisticRegression stacking
#
# Excluded on purpose:
# - LGBM meta stacking (too easy to overfit on OOF)
# =========================================================

## Imports

In [2]:
import os
import json
from collections import Counter

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import RidgeCV, LogisticRegression

## CFG

In [3]:
class CFG:
    EXP_ID = "exp_20260410_038_ridge_screen_dao037_vs_core"
    SEED = 42
    N_SPLITS = 5

    TARGET = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    # npy paths
    OOF_018 = NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"
    PRED_018 = NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"

    OOF_025 = NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy"
    PRED_025 = NPY_PATH + "pred_lgb_digit_te_min_proba_biased.npy"

    OOF_026 = NPY_PATH + "oof_realmlp_pair_te_min_proba_biased.npy"
    PRED_026 = NPY_PATH + "pred_realmlp_pair_te_min_proba_biased.npy"

    # OOF_037 = NPY_PATH + "oof_dao_xgb_pair_te_posthoc_only_proba.npy"
    # PRED_037 = NPY_PATH + "pred_dao_xgb_pair_te_posthoc_only_proba.npy"

    # OOF_039 = NPY_PATH + "oof_xgb_formula_logit_min_proba.npy"
    # PRED_039 = NPY_PATH + "pred_xgb_formula_logit_min_proba.npy"
    
    RIDGE_ALPHAS = np.logspace(-3, 3, 25)
    LR_C = 1.0
    HC_MAX_SLOTS = 25
    NM_RESTARTS = 20

## Utility

In [4]:
# ---------------------------------------------------------
# Utils
# ---------------------------------------------------------
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_proba(path):
    arr = np.load(path)
    assert arr.ndim == 2 and arr.shape[1] == 3, f"Unexpected shape {arr.shape} for {path}"
    return arr.astype(np.float32)

def rankdata_1d(x):
    return pd.Series(x).rank(method="average").to_numpy()

def mean_raw_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        vals.append(np.corrcoef(a[:, c], b[:, c])[0, 1])
    return float(np.mean(vals))

def mean_rank_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        ra = rankdata_1d(a[:, c])
        rb = rankdata_1d(b[:, c])
        vals.append(np.corrcoef(ra, rb)[0, 1])
    return float(np.mean(vals))

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

def neg_balanced_acc(raw_weights, oof_stack, y):
    w = softmax(raw_weights)
    blended = np.tensordot(w, oof_stack, axes=([0], [0]))
    preds = np.argmax(blended, axis=1)
    return -balanced_accuracy_score(y, preds)

def hill_climb(oof_stack, y, model_names, max_slots=25):
    n_models = len(model_names)
    ensemble_sum = np.zeros_like(oof_stack[0])
    selected = []
    scores_history = []

    for slot in range(max_slots):
        best_score = -1.0
        best_idx = -1

        for i in range(n_models):
            cand_sum = ensemble_sum + oof_stack[i]
            cand_avg = cand_sum / (len(selected) + 1)
            sc = balanced_accuracy_score(y, np.argmax(cand_avg, axis=1))
            if sc > best_score:
                best_score = sc
                best_idx = i

        ensemble_sum = ensemble_sum + oof_stack[best_idx]
        selected.append(best_idx)
        scores_history.append(best_score)

        print(f"  slot {slot+1:02d}: +{model_names[best_idx]} score={best_score:.6f}")

    best_slot = int(np.argmax(scores_history))
    best_score = scores_history[best_slot]
    final_selected = selected[:best_slot + 1]

    counts = Counter(final_selected)
    weights = np.zeros(n_models, dtype=np.float32)
    for idx, cnt in counts.items():
        weights[idx] = cnt / len(final_selected)

    return weights, float(best_score), int(best_slot + 1), scores_history

def multiclass_stack_features(proba_dict, keys):
    return np.hstack([proba_dict[k] for k in keys]).astype(np.float32)

def fit_ridge_oof(X, y, X_test, alphas, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_pred = np.zeros((len(X), 3), dtype=np.float32)
    test_pred = np.zeros((len(X_test), 3), dtype=np.float32)
    fold_scores = []
    alpha_info = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        va_pred = np.zeros((len(va_idx), 3), dtype=np.float32)
        te_pred = np.zeros((len(X_test), 3), dtype=np.float32)

        fold_alpha = {}

        for cls in range(3):
            y_bin = (y_tr == cls).astype(np.float32)
            model = RidgeCV(alphas=alphas)
            model.fit(X_tr, y_bin)

            va_pred[:, cls] = model.predict(X_va)
            te_pred[:, cls] = model.predict(X_test)
            fold_alpha[f"class_{cls}"] = float(model.alpha_)

        oof_pred[va_idx] = va_pred
        test_pred += te_pred / n_splits

        score = balanced_accuracy_score(y_va, np.argmax(va_pred, axis=1))
        fold_scores.append(float(score))
        alpha_info.append(fold_alpha)
        print(f"  fold {fold} ridge BA: {score:.6f}")

    return oof_pred, test_pred, fold_scores, alpha_info

def fit_lr_stack_oof(X, y, X_test, c=1.0, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_pred = np.zeros((len(X), 3), dtype=np.float32)
    test_pred = np.zeros((len(X_test), 3), dtype=np.float32)
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model = LogisticRegression(
            max_iter=2000,
            C=c,
            solver="lbfgs",
            multi_class="auto",
            class_weight="balanced",
            random_state=seed,
        )
        model.fit(X_tr, y_tr)

        oof_pred[va_idx] = model.predict_proba(X_va)
        test_pred += model.predict_proba(X_test) / n_splits

        score = balanced_accuracy_score(y_va, np.argmax(oof_pred[va_idx], axis=1))
        fold_scores.append(float(score))
        print(f"  fold {fold} lr-stack BA: {score:.6f}")

    return oof_pred, test_pred, fold_scores

## Load data

In [5]:
# ---------------------------------------------------------
# Load
# ---------------------------------------------------------
ensure_dir(CFG.OUTPUT_DIR)

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
y = train[CFG.TARGET].map(CFG.TARGET_MAP).values

proba_oof = {
    "018": load_proba(CFG.OOF_018),
    "025": load_proba(CFG.OOF_025),
    "026": load_proba(CFG.OOF_026),
}

proba_pred = {
    "018": load_proba(CFG.PRED_018),
    "025": load_proba(CFG.PRED_025),
    "026": load_proba(CFG.PRED_026),
}

model_names = ["018", "025", "026"]
oof_stack = np.array([proba_oof[n] for n in model_names], dtype=np.float32)
test_stack = np.array([proba_pred[n] for n in model_names], dtype=np.float32)

## Correlation summary

In [6]:
# ---------------------------------------------------------
# Correlation summary
# ---------------------------------------------------------
corr_rows = []
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        a = model_names[i]
        b = model_names[j]
        corr_rows.append({
            "a": a,
            "b": b,
            "raw_corr_mean": mean_raw_corr(proba_oof[a], proba_oof[b]),
            "rank_corr_mean": mean_rank_corr(proba_oof[a], proba_oof[b]),
        })

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(f"{CFG.OUTPUT_DIR}/pairwise_corr_summary.csv", index=False)

## Single baselines

In [7]:
# ---------------------------------------------------------
# Single baselines
# ---------------------------------------------------------
single_rows = []
for name in model_names:
    sc = balanced_accuracy_score(y, np.argmax(proba_oof[name], axis=1))
    single_rows.append({
        "strategy": f"single_{name}",
        "cv_score": float(sc),
    })

single_df = pd.DataFrame(single_rows).sort_values("cv_score", ascending=False)
single_df.to_csv(f"{CFG.OUTPUT_DIR}/single_model_scores.csv", index=False)


## Equal average

In [8]:
# ---------------------------------------------------------
# Equal average
# ---------------------------------------------------------
avg_oof = oof_stack.mean(axis=0)
avg_test = test_stack.mean(axis=0)
avg_score = balanced_accuracy_score(y, np.argmax(avg_oof, axis=1))


## Nelder-Mead

In [9]:
# ---------------------------------------------------------
# Nelder-Mead
# ---------------------------------------------------------
best_result = None
best_obj = None

for restart in range(CFG.NM_RESTARTS):
    rng = np.random.RandomState(CFG.SEED + restart)
    x0 = rng.randn(len(model_names)) * 0.5
    result = minimize(
        neg_balanced_acc,
        x0,
        args=(oof_stack, y),
        method="Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-7, "fatol": 1e-7},
    )
    if best_result is None or result.fun < best_obj:
        best_result = result
        best_obj = result.fun

nm_weights = softmax(best_result.x)
nm_oof = np.tensordot(nm_weights, oof_stack, axes=([0], [0]))
nm_test = np.tensordot(nm_weights, test_stack, axes=([0], [0]))
nm_score = balanced_accuracy_score(y, np.argmax(nm_oof, axis=1))

nm_weight_df = pd.DataFrame({
    "model": model_names,
    "weight": nm_weights,
}).sort_values("weight", ascending=False)
nm_weight_df.to_csv(f"{CFG.OUTPUT_DIR}/nelder_mead_weights.csv", index=False)

## Greedy hill climb

In [10]:
# ---------------------------------------------------------
# Greedy hill climb
# ---------------------------------------------------------
hc_weights, hc_best_score, hc_n_slots, hc_history = hill_climb(
    oof_stack=oof_stack,
    y=y,
    model_names=model_names,
    max_slots=CFG.HC_MAX_SLOTS,
)

hc_oof = np.tensordot(hc_weights, oof_stack, axes=([0], [0]))
hc_test = np.tensordot(hc_weights, test_stack, axes=([0], [0]))
hc_score = balanced_accuracy_score(y, np.argmax(hc_oof, axis=1))

hc_weight_df = pd.DataFrame({
    "model": model_names,
    "weight": hc_weights,
}).sort_values("weight", ascending=False)
hc_weight_df.to_csv(f"{CFG.OUTPUT_DIR}/hill_climb_weights.csv", index=False)

pd.DataFrame({
    "slot": list(range(1, len(hc_history) + 1)),
    "score": hc_history,
}).to_csv(f"{CFG.OUTPUT_DIR}/hill_climb_history.csv", index=False)

  slot 01: +025 score=0.979396
  slot 02: +026 score=0.979856
  slot 03: +025 score=0.979861
  slot 04: +018 score=0.980014
  slot 05: +025 score=0.979941
  slot 06: +025 score=0.979967
  slot 07: +018 score=0.979993
  slot 08: +026 score=0.980014
  slot 09: +026 score=0.979971
  slot 10: +018 score=0.979952
  slot 11: +025 score=0.980009
  slot 12: +026 score=0.980022
  slot 13: +025 score=0.979953
  slot 14: +025 score=0.979994
  slot 15: +018 score=0.980003
  slot 16: +025 score=0.980014
  slot 17: +018 score=0.979993
  slot 18: +026 score=0.980005
  slot 19: +025 score=0.980003
  slot 20: +025 score=0.980014
  slot 21: +018 score=0.980017
  slot 22: +026 score=0.980009
  slot 23: +026 score=0.980026
  slot 24: +026 score=0.980022
  slot 25: +025 score=0.979961


## Ridge blend

In [11]:
# ---------------------------------------------------------
# Ridge blend
# ---------------------------------------------------------
stack_feats_oof = multiclass_stack_features(proba_oof, model_names)
stack_feats_test = multiclass_stack_features(proba_pred, model_names)

ridge_oof, ridge_test, ridge_fold_scores, ridge_alpha_info = fit_ridge_oof(
    X=stack_feats_oof,
    y=y,
    X_test=stack_feats_test,
    alphas=CFG.RIDGE_ALPHAS,
    n_splits=CFG.N_SPLITS,
    seed=CFG.SEED,
)
ridge_score = balanced_accuracy_score(y, np.argmax(ridge_oof, axis=1))

pd.DataFrame(ridge_alpha_info).to_csv(f"{CFG.OUTPUT_DIR}/ridge_alpha_summary.csv", index=False)

  fold 1 ridge BA: 0.978752
  fold 2 ridge BA: 0.978706
  fold 3 ridge BA: 0.980654
  fold 4 ridge BA: 0.979974
  fold 5 ridge BA: 0.980028


## LogisticRegression stacking

In [12]:
# ---------------------------------------------------------
# LogisticRegression stacking
# ---------------------------------------------------------
lr_oof, lr_test, lr_fold_scores = fit_lr_stack_oof(
    X=stack_feats_oof,
    y=y,
    X_test=stack_feats_test,
    c=CFG.LR_C,
    n_splits=CFG.N_SPLITS,
    seed=CFG.SEED,
)
lr_score = balanced_accuracy_score(y, np.argmax(lr_oof, axis=1))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  fold 1 lr-stack BA: 0.978539


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  fold 2 lr-stack BA: 0.979205


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  fold 3 lr-stack BA: 0.980456


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  fold 4 lr-stack BA: 0.980256


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  fold 5 lr-stack BA: 0.980093


## Summary

In [13]:
# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------
results = [
    {"strategy": "single_018", "cv_score": balanced_accuracy_score(y, np.argmax(proba_oof["018"], axis=1))},
    {"strategy": "single_025", "cv_score": balanced_accuracy_score(y, np.argmax(proba_oof["025"], axis=1))},
    {"strategy": "single_026", "cv_score": balanced_accuracy_score(y, np.argmax(proba_oof["026"], axis=1))},
    {"strategy": "equal_avg", "cv_score": avg_score},
    {"strategy": "nelder_mead", "cv_score": nm_score},
    {"strategy": "hill_climb", "cv_score": hc_score},
    {"strategy": "ridge", "cv_score": ridge_score},
    {"strategy": "stack_lr", "cv_score": lr_score},
]

results_df = pd.DataFrame(results).sort_values("cv_score", ascending=False).reset_index(drop=True)
best_single_025 = float(results_df.loc[results_df["strategy"] == "single_025", "cv_score"].iloc[0])
results_df["gain_vs_single_025"] = results_df["cv_score"] - best_single_025
results_df.to_csv(f"{CFG.OUTPUT_DIR}/blend_strategy_results.csv", index=False)

## Save OOF/Test/Submission per strategy

In [14]:
# ---------------------------------------------------------
# Save OOF/Test/Submission per strategy
# ---------------------------------------------------------
strategy_oof = {
    "equal_avg": avg_oof,
    "nelder_mead": nm_oof,
    "hill_climb": hc_oof,
    "ridge": ridge_oof,
    "stack_lr": lr_oof,
}
strategy_test = {
    "equal_avg": avg_test,
    "nelder_mead": nm_test,
    "hill_climb": hc_test,
    "ridge": ridge_test,
    "stack_lr": lr_test,
}

for name, arr in strategy_oof.items():
    np.save(f"{CFG.OUTPUT_DIR}/oof_{name}.npy", arr)

for name, arr in strategy_test.items():
    np.save(f"{CFG.OUTPUT_DIR}/pred_{name}.npy", arr)

    sub = pd.DataFrame({
        CFG.ID_COL: test[CFG.ID_COL],
        CFG.TARGET: pd.Series(np.argmax(arr, axis=1)).map(CFG.INV_TARGET_MAP)
    })
    sub.to_csv(f"{CFG.OUTPUT_DIR}/submission_{name}.csv", index=False)

summary = {
    "exp_id": CFG.EXP_ID,
    "seed": CFG.SEED,
    "n_splits": CFG.N_SPLITS,
    "strategies_ranked": results_df.to_dict(orient="records"),
    "nm_weights": dict(zip(model_names, nm_weights.tolist())),
    "hc_weights": dict(zip(model_names, hc_weights.tolist())),
    "ridge_fold_scores": ridge_fold_scores,
    "lr_fold_scores": lr_fold_scores,
    "notes": {
        "safe_mode": True,
        "excluded_strategy": "stack_lgb",
        "reason": "nonlinear meta learner is more likely to overfit on OOF in this setting",
    },
}
save_json(summary, f"{CFG.OUTPUT_DIR}/cv_result.json")

print(results_df)
print("\nSaved to:", CFG.OUTPUT_DIR)

      strategy  cv_score  gain_vs_single_025
0  nelder_mead  0.980028            0.000631
1   hill_climb  0.980026            0.000630
2    equal_avg  0.979845            0.000448
3     stack_lr  0.979710            0.000313
4        ridge  0.979623            0.000227
5   single_025  0.979396            0.000000
6   single_018  0.979091           -0.000305
7   single_026  0.978113           -0.001284

Saved to: /kaggle/working/exp_20260410_038_ridge_screen_dao037_vs_core
